In [1]:
#Make sure R can find packages installed with Conda
.libPaths('/hb/home/jbos/.conda/envs/vcfR')
.libPaths("/hb/home/jbos/.conda/envs/vcfR/lib/R/library")

In [2]:
#Load packages
library(tidyverse)
library(vcfR)
library(adegenet)
library(hierfstat)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

   *****       ***   vcfR   ***       *****
   This is vcfR 1.15.0 
     browseVignettes('vcfR') # Documentation
     citation('vcfR') # Citation
   *****       *****      *****       *****


Loading required package: ade4


   /// adegenet 2.1.11 is loaded ////////////

   > overview: '?adegenet'
   > tutorials/doc/questions: 'adegenetWeb()' 
   > bug reports/feature requests: adegenetIssues()




Attaching package: ‘hierfstat’


The following objects are masked from ‘p

In [3]:
#Load in SNP data in VCF format
all_snps <- read.vcfR("vcfs/pruned_snps_all.vcf")

Scanning file to determine attributes.
File attributes:
  meta lines: 289
  header_line: 290
  variant count: 1838
  column count: 310
Meta line 289 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 1838
  Character matrix gt cols: 310
  skip: 0
  nrows: 1838
  row_num: 0
Processed variant: 1838
All variants processed


In [4]:
#Convert to genind format
genind_snps <- vcfR2genind(all_snps)

In [5]:
inds1 = read.table('other_data/taxa1_dfa.txt')[,1]
inds2 = read.table('other_data/taxa2_dfa.txt')[,1]
inds3 = read.table('other_data/taxa3_dfa.txt')[,1]
inds4 = read.table('other_data/taxa4_dfa.txt')[,1]

In [6]:
#Make dataframe associating individual samples to the correct taxa
individuals<-as.data.frame(matrix(nrow=nrow(genind_snps@tab),ncol=1))
individuals$ind<-rownames(genind_snps@tab)
colnames(individuals)<-c('taxa','ind')

individuals$taxa<-0
individuals[individuals$ind %in% inds1,1]<-1
individuals[individuals$ind %in% inds2,1]<-2
individuals[individuals$ind %in% inds3,1]<-3
individuals[individuals$ind %in% inds4,1]<-4

In [7]:
#Add population variable to the genind (taxon=population)
genind_snps@pop<-as.factor(individuals$taxa)

In [8]:
#Convert to hierfstat
hstat2<-genind2hierfstat(genind_snps,pop=genind_snps@pop)

In [9]:
#Calculate pairwise FSTs based on cryptic Acropora taxa
fst_acropora<-pairwise.WCfst(hstat2)

In [10]:
#Print FSTs
fst_acropora

,1,2,3,4
1,NA,0.11777279,0.11045811,0.1568439
2,0.1177728,NA,0.09925483,0.1476028
3,0.1104581,0.09925483,NA,0.1210137
4,0.1568439,0.14760283,0.12101369,NA


In [14]:
#Read in bootstrapped distributions of null FSTs between reshuffled taxa
boot12<-ecdf(read.csv('acropora12_fst_boot.csv')[,2])
boot13<-ecdf(read.csv('acropora13_fst_boot.csv')[,2])
boot14<-ecdf(read.csv('acropora14_fst_boot.csv')[,2])
boot23<-ecdf(read.csv('acropora23_fst_boot.csv')[,2])
boot24<-ecdf(read.csv('acropora24_fst_boot.csv')[,2])
boot34<-ecdf(read.csv('acropora34_fst_boot.csv')[,2])

In [15]:
#Check simulated significance of FST between taxa 1 and 2
1- boot12(fst_acropora[1,2])

[1] 0

In [16]:
#Check simulated significance of FST between taxa 1 and 3
1- boot13(fst_acropora[1,3])

[1] 0

In [17]:
#Check simulated significance of FST between taxa 1 and 4
1- boot14(fst_acropora[1,4])

[1] 0

In [18]:
#Check simulated significance of FST between taxa 2 and 3
1- boot23(fst_acropora[2,3])

[1] 0

In [19]:
#Check simulated significance of FST between taxa 2 and 4
1- boot24(fst_acropora[2,4])

[1] 0

In [20]:
#Check simulated significance of FST between taxa 3 and 4
1- boot34(fst_acropora[3,4])

[1] 0